# Escucha ciudadana — septiembre y octubre de 2026

Análisis de los registros del **01/09/2026 al 25/10/2026 inclusive**. Octubre es parcial.

Entradas verificadas: **10.743 comentarios públicos**, **7.571 comentarios de encuestas** y **299.011 mensajes privados**. Total: **317.325 registros reales**, sin muestreo ni ponderación. No son personas ni conversaciones únicas. El universo privado contiene texto con al menos una letra, con las exclusiones documentadas de pruebas, spam, borrados y marcadores de adjuntos.

El sentimiento se calcula a partir del texto ciudadano, no del NPS ni de las respuestas del bot. Las temáticas pueden provenir del texto o de etiquetas del caso; se conserva esa distinción. La anonimización automática es básica: revisar datos personales y la muestra humana antes de compartir o publicar. Las encuestas se ubican temporalmente por fecha de envío, no necesariamente por fecha de respuesta.

**Seleccionar Entorno de ejecución → Cambiar tipo de entorno → GPU.** No subir los archivos de agosto ni la muestra privada anterior. Los scripts ya están incluidos en este notebook.


## 1. Preparar el entorno

Esta celda instala dependencias. Si Colab solicita reiniciar la sesión, reiniciarla y continuar desde la celda 2.


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'pandas>=2.2,<3', 'numpy>=1.26,<3',
                'transformers>=4.44,<5', 'torch>=2.3,<3', 'accelerate>=0.33,<2'], check=True)


## 2. Subir los datos

Seleccionar **datos_colab_sep_oct_2026.zip**, sin descomprimirlo. Opcionalmente se puede subir también una caché `.sqlite` descargada de una ejecución interrumpida de este mismo notebook. No se necesita subir ningún `.py`.


In [ ]:
from google.colab import files
from pathlib import Path
import json
import shutil
import zipfile

uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Seleccionar un único ZIP de datos de septiembre/octubre.')
DATA = Path('/content/datos_sep_oct_2026')
DATA.mkdir(parents=True, exist_ok=True)
expected = {'social_publico.csv', 'encuestas.csv', 'privados_completos.csv',
            'control_totales.csv', 'manifest_inputs.json'}
with zipfile.ZipFile(archives[0]) as archive:
    if len(archive.namelist()) != len(expected) or set(archive.namelist()) != expected:
        raise ValueError('El ZIP no contiene exactamente los archivos esperados.')
    for member in archive.infolist():
        if member.file_size > 500_000_000:
            raise ValueError('Archivo inesperadamente grande en el ZIP.')
    archive.extractall(DATA)
manifest = json.loads((DATA/'manifest_inputs.json').read_text(encoding='utf-8'))
if (manifest['expected_total'] != 317325 or manifest['start_inclusive'] != '2026-09-01'
    or manifest['end_exclusive'] != '2026-10-26' or manifest['sampling'] or manifest['weighting']):
    raise ValueError('El paquete no corresponde al período y método acordados.')
CACHE = Path('/content/cache_sentimiento_sep_oct_2026.sqlite')
cache_files = [name for name in uploaded if name.lower().endswith('.sqlite')]
if len(cache_files) > 1:
    raise ValueError('Subir como máximo una caché de sentimiento.')
if cache_files and Path(cache_files[0]).resolve() != CACHE.resolve():
    shutil.copyfile(cache_files[0], CACHE)
del uploaded
print(json.dumps(manifest, ensure_ascii=False, indent=2))


## 3. Crear el proceso reproducible

Ejecutar las tres celdas siguientes. La revisión del modelo se fija a la misma usada en la corrida previa. Se mantienen su normalización y límite de 128 tokens; los textos largos pueden perder contexto.


In [ ]:
%%writefile /content/analisis_escucha_ciudadana_v2.py
from __future__ import annotations

import argparse
import html
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


MODEL_NAME = "pysentimiento/robertuito-sentiment-analysis"
CONTROL_TERMS = {
    "hola", "holaa", "holaaa", "si", "no", "ok", "oki", "dale",
    "continuar", "volver", "menu", "buen dia", "buenas", "a",
}

EMAIL_RE = re.compile(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", re.I)
URL_RE = re.compile(r"(?:https?://|www\.)\S+", re.I)
LONG_NUMBER_RE = re.compile(r"(?<!\d)(?:\+?\d[\s().-]*){8,}(?!\d)")
HTML_TAG_RE = re.compile(r"<[^>]+>")

THEME_PATTERNS = [
    ("Residuos y limpieza", r"residu|basura|contenedor|recoleccion|limpieza|barrido|recicla"),
    ("Calles y veredas", r"bache|pozo|vereda|cordon|pavimento|calzada|adoquin"),
    ("Alumbrado público", r"alumbr|luminaria|farola|poste de luz|luz de la calle|luces de la calle"),
    ("Arbolado", r"arbol|poda|rama|raiz|forestacion"),
    ("Anegamiento y desagües", r"aneg|inund|desague|alcantar|sumidero|pluvial"),
    ("Estacionamiento y tránsito", r"estaciona|transito|semaforo|cochera|vehiculo abandonado|corte de calle"),
    ("Transporte público", r"colectiv|transporte|sube|taxi|remis|parada|linea \d"),
    ("Bicicletas", r"bici|bicicleta|mibici|ciclovia"),
    ("Ruidos molestos", r"ruido|musica fuerte|volumen alto"),
    ("Personas en situación de calle", r"situacion de calle|persona.*calle|sin techo"),
    ("Seguridad", r"seguridad|robo|delinc|insegur|policia|violencia"),
    ("Obras y espacio público", r"obra|plaza|parque|rambla|espacio publico|juego infantil"),
    ("Licencia de conducir", r"licencia|carnet de conducir"),
    ("Multas y liberación de vehículos", r"multa|liberacion|corralon|remision|acarreo"),
    ("Tributos y TGI", r"\btgi\b|tribut|impuesto|tasa municipal|pago anticipado"),
    ("Habilitaciones y comercio", r"habilit|comercio|industria|fiscalizacion ambiental|instituto.*alimento"),
    ("Salud y bienestar animal", r"salud|hospital|centro de salud|vacuna|animal|veterin|mascota"),
    ("Agenda, cultura y deportes", r"agenda|cultura|evento|festival|deporte|pileta|colectividad"),
    ("MuniBot y atención digital", r"munibot|\bbot\b|whatsapp|pagina|sitio web|web municipal|perfil digital|firma digital"),
    ("Gestión municipal e Intendente", r"municipalidad|municipio|intendente|javkin|gestion municipal"),
]

COMPLAINT_RE = re.compile(
    r"reclamo|denuncia|verguenza|desastre|no funciona|no anda|falta|roto|"
    r"mal estado|abandono|sucio|basura|pozo|inund|demora|nadie responde|problema"
)
QUESTION_RE = re.compile(r"\?|como |cuando |donde |cual |quisiera saber|consult")
REQUEST_RE = re.compile(r"por favor|necesitamos|necesito|deberian|solicito|pedido|quiero que|hace falta")
THANKS_RE = re.compile(r"gracias|agradezco|agradecimiento")
PRAISE_RE = re.compile(r"felicit|excelente|hermos|muy bueno|maravill|gran trabajo")
FRUSTRATION_RE = re.compile(
    r"harto|cansad|indign|bronca|verg[uü]enza|desastre|inutil|nunca|siempre lo mismo|"
    r"nadie responde|abandono|una porqueria|caradura"
)

SPANISH_STOPWORDS = {
    "a", "al", "algo", "ante", "como", "con", "contra", "cual", "cuando", "de", "del",
    "desde", "donde", "el", "ella", "ellos", "en", "entre", "era", "es", "esa", "ese",
    "eso", "esta", "este", "esto", "fue", "ha", "hay", "la", "las", "le", "les", "lo",
    "los", "mas", "me", "mi", "muy", "no", "nos", "o", "para", "pero", "por", "porque",
    "que", "se", "si", "sin", "sobre", "son", "su", "sus", "te", "tiene", "todo", "un",
    "una", "uno", "y", "ya",
}


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Prototipo reproducible de escucha ciudadana MuniBot")
    parser.add_argument("--social", required=True)
    parser.add_argument("--surveys", required=True)
    parser.add_argument("--conversations", required=True)
    parser.add_argument("--weights", required=True)
    parser.add_argument("--output-dir", default="salidas_escucha_ciudadana")
    parser.add_argument("--skip-sentiment", action="store_true")
    parser.add_argument("--skip-clusters", action="store_true")
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--confidence-threshold", type=float, default=0.55)
    parser.add_argument("--clusters", type=int, default=20)
    parser.add_argument("--random-state", type=int, default=42)
    return parser.parse_args()


def read_source(path: str) -> pd.DataFrame:
    return pd.read_csv(path, dtype=str, keep_default_na=False, low_memory=False)


def remove_accents(value: str) -> str:
    return "".join(
        character for character in unicodedata.normalize("NFD", value)
        if unicodedata.category(character) != "Mn"
    )


def normalize_text(value: str) -> str:
    value = remove_accents(str(value or "").lower())
    value = URL_RE.sub(" [url] ", value)
    value = re.sub(r"[^a-z0-9ñ\[\]]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def anonymize_text(value: str) -> tuple[str, bool]:
    value = html.unescape(str(value or ""))
    contains_pii = bool(EMAIL_RE.search(value) or LONG_NUMBER_RE.search(value))
    value = EMAIL_RE.sub("[EMAIL]", value)
    value = URL_RE.sub("[URL]", value)
    value = LONG_NUMBER_RE.sub("[NÚMERO]", value)
    value = HTML_TAG_RE.sub(" ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value, contains_pii


def build_unified_dataset(
    social: pd.DataFrame,
    surveys: pd.DataFrame,
    conversations: pd.DataFrame,
    weights: pd.DataFrame,
) -> pd.DataFrame:
    social_dates = pd.to_datetime(social["created_at"], dayfirst=True, errors="coerce")
    social_out = pd.DataFrame({
        "record_id": "SOC_" + social["activity_id"],
        "case_id": social["case_id"],
        "contact_hash": social["contacto_anonimo"],
        "created_at": social_dates,
        "source_group": "Social público",
        "source_detail": social["red_social"] + " - " + social["origen_publicacion"],
        "source_channel": social["source_channel"],
        "channel": social["canal"],
        "survey_name": "",
        "nps_value": np.nan,
        "raw_text": social["texto"],
        "raw_context": "",
        "tags": social["tags"],
        "sample_weight": 1.0,
        "is_sample": False,
    })

    survey_dates = pd.to_datetime(surveys["sent_at"], dayfirst=True, errors="coerce")
    survey_nps = pd.to_numeric(surveys["nps_value"], errors="coerce")
    survey_out = pd.DataFrame({
        "record_id": "ENC_" + surveys["response_id"],
        "case_id": surveys["case_id"],
        "contact_hash": surveys["contacto_anonimo"],
        "created_at": survey_dates,
        "source_group": "Encuesta",
        "source_detail": surveys["survey_name"],
        "source_channel": surveys["source_channel"],
        "channel": surveys["source_channel"],
        "survey_name": surveys["survey_name"],
        "nps_value": survey_nps,
        "raw_text": surveys["comentario"],
        "raw_context": "",
        "tags": surveys["tags"],
        "sample_weight": 1.0,
        "is_sample": False,
    })

    conversation_dates = pd.to_datetime(conversations["created_at"], dayfirst=True, errors="coerce")
    conversations = conversations.copy()
    conversations["month_key"] = conversation_dates.dt.strftime("%Y-%m")

    weights = weights.copy()
    weights["month_key"] = pd.to_datetime(weights["mes"], dayfirst=True, errors="coerce").dt.strftime("%Y-%m")
    weights["population_messages"] = pd.to_numeric(
        weights["mensajes_poblacion"].str.replace(".", "", regex=False), errors="coerce"
    )
    sample_counts = (
        conversations.groupby(["month_key", "source_channel"], dropna=False)
        .size()
        .rename("sample_messages")
        .reset_index()
    )
    weight_table = weights.merge(sample_counts, on=["month_key", "source_channel"], how="left", validate="one_to_one")
    weight_table["sample_weight"] = weight_table["population_messages"] / weight_table["sample_messages"]
    if weight_table["sample_weight"].isna().any():
        missing = weight_table.loc[weight_table["sample_weight"].isna(), ["month_key", "source_channel"]]
        raise ValueError(f"Faltan ponderaciones para: {missing.to_dict('records')}")

    conversations = conversations.merge(
        weight_table[["month_key", "source_channel", "sample_weight"]],
        on=["month_key", "source_channel"],
        how="left",
        validate="many_to_one",
    )
    if conversations["sample_weight"].isna().any():
        raise ValueError("Existen conversaciones sin ponderación")

    conversation_out = pd.DataFrame({
        "record_id": "CON_" + conversations["activity_id"],
        "case_id": conversations["case_id"],
        "contact_hash": conversations["contacto_anonimo"],
        "created_at": conversation_dates,
        "source_group": "Conversación privada",
        "source_detail": conversations["source_channel"],
        "source_channel": conversations["source_channel"],
        "channel": conversations["channel"],
        "survey_name": "",
        "nps_value": np.nan,
        "raw_text": conversations["mensaje_ciudadano"],
        "raw_context": conversations["contexto_bot_anterior"],
        "tags": conversations["tags"],
        "sample_weight": conversations["sample_weight"].astype(float),
        "is_sample": True,
    })

    unified = pd.concat([social_out, survey_out, conversation_out], ignore_index=True)
    if unified["record_id"].duplicated().any():
        raise ValueError("Se detectaron record_id duplicados")
    unified["month"] = unified["created_at"].dt.to_period("M").astype(str)
    return unified


def classify_text_type(raw_text: str, normalized_text: str, word_count: int) -> str:
    raw_lower = str(raw_text or "").lower()
    if re.search(r"ha enviado (?:una|un).*?(?:imagen|documento|archivo|audio|video)", raw_lower):
        return "Adjunto o archivo"
    if re.fullmatch(r"\d+", normalized_text):
        return "Opción numérica"
    if normalized_text in CONTROL_TERMS:
        return "Saludo o control"
    if normalized_text in {"url", "[url]"} or ("[url]" in normalized_text and word_count <= 3):
        return "Enlace sin contenido"
    if word_count <= 2:
        return "Texto breve"
    return "Contenido"


def detect_themes(normalized_text: str, normalized_tags: str) -> tuple[str, str]:
    combined = f"{normalized_text} {normalized_tags}"
    detected = [name for name, pattern in THEME_PATTERNS if re.search(pattern, combined)]
    if not detected:
        return "Otros / por descubrir", ""
    return detected[0], " | ".join(detected)


def detect_intents(raw_text: str, normalized_text: str) -> tuple[str, str]:
    intents = []
    if COMPLAINT_RE.search(normalized_text):
        intents.append("Reclamo o problema")
    if QUESTION_RE.search(normalized_text) or "?" in str(raw_text or ""):
        intents.append("Consulta")
    if REQUEST_RE.search(normalized_text):
        intents.append("Pedido o sugerencia")
    if THANKS_RE.search(normalized_text):
        intents.append("Agradecimiento")
    if PRAISE_RE.search(normalized_text):
        intents.append("Reconocimiento")
    if not intents:
        intents.append("Comentario u otro")
    return intents[0], " | ".join(intents)


def add_quality_and_rules(dataset: pd.DataFrame) -> pd.DataFrame:
    dataset = dataset.copy()
    anonymized = dataset["raw_text"].map(anonymize_text)
    dataset["text_anonymized"] = anonymized.map(lambda value: value[0])
    dataset["pii_detected"] = anonymized.map(lambda value: value[1])
    context_anonymized = dataset["raw_context"].map(anonymize_text)
    dataset["context_anonymized"] = context_anonymized.map(lambda value: value[0])
    dataset["context_pii_detected"] = context_anonymized.map(lambda value: value[1])

    dataset["text_normalized"] = dataset["text_anonymized"].map(normalize_text)
    dataset["tags_normalized"] = dataset["tags"].map(normalize_text)
    dataset["word_count"] = dataset["text_anonymized"].str.findall(
        r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]+"
    ).str.len()
    dataset["text_type"] = [
        classify_text_type(raw, normalized, int(words))
        for raw, normalized, words in zip(dataset["raw_text"], dataset["text_normalized"], dataset["word_count"])
    ]

    blocked_sentiment = {"Adjunto o archivo", "Opción numérica", "Saludo o control", "Enlace sin contenido"}
    dataset["eligible_sentiment"] = ~dataset["text_type"].isin(blocked_sentiment)
    dataset["eligible_theme"] = (dataset["text_type"] == "Contenido") & (dataset["word_count"] >= 3)

    repetitions = dataset.groupby(["source_group", "text_normalized"])["record_id"].transform("size")
    repetitions = repetitions.where(dataset["text_normalized"].ne(""), 0)
    dataset["text_repetitions"] = repetitions.astype(int)
    dataset["repeated_text"] = dataset["text_repetitions"] > 1
    dataset["possible_campaign"] = (
        dataset["source_group"].eq("Social público") & (dataset["text_repetitions"] >= 20)
    )

    themes = [
        detect_themes(text_value, tags_value)
        for text_value, tags_value in zip(dataset["text_normalized"], dataset["tags_normalized"])
    ]
    dataset["primary_theme"] = [value[0] for value in themes]
    dataset["detected_themes"] = [value[1] for value in themes]

    intents = [
        detect_intents(raw_value, normalized_value)
        for raw_value, normalized_value in zip(dataset["raw_text"], dataset["text_normalized"])
    ]
    dataset["primary_intent"] = [value[0] for value in intents]
    dataset["detected_intents"] = [value[1] for value in intents]

    upper_ratio = dataset["raw_text"].map(
        lambda value: sum(character.isupper() for character in str(value))
        / max(sum(character.isalpha() for character in str(value)), 1)
    )
    dataset["frustration_signal"] = (
        dataset["text_normalized"].str.contains(FRUSTRATION_RE, regex=True)
        | ((upper_ratio >= 0.60) & (dataset["word_count"] >= 4))
        | dataset["raw_text"].str.contains(r"!{2,}|\?{3,}", regex=True)
    )
    return dataset


def add_sentiment(
    dataset: pd.DataFrame,
    batch_size: int,
    confidence_threshold: float,
) -> tuple[pd.DataFrame, dict]:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    eligible = (
        dataset.loc[dataset["eligible_sentiment"], "text_normalized"]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    config_labels = [str(model.config.id2label[index]).upper() for index in range(model.config.num_labels)]
    if not set(config_labels).issubset({"POS", "NEG", "NEU"}):
        config_labels = ["NEG", "NEU", "POS"]

    predictions = []
    with torch.inference_mode():
        for start in range(0, len(eligible), batch_size):
            batch = eligible.iloc[start:start + batch_size].tolist()
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt",
            ).to(device)
            probabilities = torch.softmax(model(**encoded).logits, dim=-1).cpu().numpy()
            for text_value, row in zip(batch, probabilities):
                values = dict(zip(config_labels, row.tolist()))
                raw_label = max(values, key=values.get)
                confidence = values[raw_label]
                predictions.append({
                    "text_normalized": text_value,
                    "sentiment_raw": raw_label,
                    "sentiment_confidence": confidence,
                    "prob_negative": values.get("NEG", 0.0),
                    "prob_neutral": values.get("NEU", 0.0),
                    "prob_positive": values.get("POS", 0.0),
                })

    prediction_table = pd.DataFrame(predictions)
    dataset = dataset.merge(prediction_table, on="text_normalized", how="left", validate="many_to_one")
    label_map = {"POS": "Positivo", "NEG": "Negativo", "NEU": "Neutro"}
    dataset["sentiment"] = dataset["sentiment_raw"].map(label_map)
    dataset.loc[~dataset["eligible_sentiment"], "sentiment"] = "No analizable"
    dataset.loc[
        dataset["eligible_sentiment"] & (dataset["sentiment_confidence"] < confidence_threshold),
        "sentiment",
    ] = "Incierto"
    dataset["sentiment_model"] = MODEL_NAME
    dataset["sentiment_model_commit"] = getattr(model.config, "_commit_hash", "") or ""

    survey_mask = dataset["source_group"].eq("Encuesta") & dataset["nps_value"].notna()
    dataset["nps_text_contradiction"] = False
    dataset.loc[
        survey_mask
        & dataset["nps_value"].isin([4, 5])
        & dataset["sentiment"].eq("Negativo"),
        "nps_text_contradiction",
    ] = True
    dataset.loc[
        survey_mask
        & dataset["nps_value"].isin([1, 2])
        & dataset["sentiment"].eq("Positivo"),
        "nps_text_contradiction",
    ] = True

    metadata = {
        "model": MODEL_NAME,
        "commit": getattr(model.config, "_commit_hash", "") or "",
        "device": str(device),
        "batch_size": batch_size,
        "confidence_threshold": confidence_threshold,
        "unique_texts_scored": len(prediction_table),
    }
    return dataset, metadata


def add_discovery_clusters(
    dataset: pd.DataFrame,
    n_clusters: int,
    random_state: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    from sklearn.cluster import MiniBatchKMeans
    from sklearn.feature_extraction.text import TfidfVectorizer

    mask = dataset["eligible_theme"] & dataset["primary_theme"].eq("Otros / por descubrir")
    unique_other = dataset.loc[mask, ["text_normalized", "source_group"]].drop_duplicates("text_normalized")
    if len(unique_other) < 100:
        dataset["discovery_cluster"] = pd.NA
        dataset["discovery_terms"] = ""
        return dataset, pd.DataFrame(columns=["cluster", "top_terms", "unique_texts"])

    fit_sample = (
        unique_other.groupby("source_group", group_keys=False)
        .apply(lambda frame: frame.sample(min(len(frame), 10000), random_state=random_state), include_groups=False)
        .reset_index(drop=True)
    )
    effective_clusters = min(n_clusters, max(2, len(fit_sample) // 300))
    vectorizer = TfidfVectorizer(
        min_df=5,
        max_df=0.85,
        ngram_range=(1, 2),
        max_features=20000,
        stop_words=sorted(SPANISH_STOPWORDS),
        sublinear_tf=True,
    )
    fit_matrix = vectorizer.fit_transform(fit_sample["text_normalized"])
    cluster_model = MiniBatchKMeans(
        n_clusters=effective_clusters,
        random_state=random_state,
        batch_size=1024,
        n_init="auto",
    )
    cluster_model.fit(fit_matrix)

    all_matrix = vectorizer.transform(unique_other["text_normalized"])
    unique_other = unique_other.copy()
    unique_other["discovery_cluster"] = cluster_model.predict(all_matrix)

    terms = np.asarray(vectorizer.get_feature_names_out())
    catalog_rows = []
    for cluster in range(effective_clusters):
        top_indices = cluster_model.cluster_centers_[cluster].argsort()[-12:][::-1]
        top_terms = " | ".join(terms[top_indices])
        catalog_rows.append({
            "cluster": cluster,
            "top_terms": top_terms,
            "unique_texts": int((unique_other["discovery_cluster"] == cluster).sum()),
        })
    catalog = pd.DataFrame(catalog_rows)
    unique_other = unique_other.merge(catalog[["cluster", "top_terms"]], left_on="discovery_cluster", right_on="cluster")
    dataset = dataset.merge(
        unique_other[["text_normalized", "discovery_cluster", "top_terms"]],
        on="text_normalized",
        how="left",
        validate="many_to_one",
    )
    dataset = dataset.rename(columns={"top_terms": "discovery_terms"})
    return dataset, catalog


def create_quality_summary(dataset: pd.DataFrame) -> pd.DataFrame:
    return (
        dataset.groupby("source_group", dropna=False)
        .agg(
            records=("record_id", "size"),
            unique_cases=("case_id", "nunique"),
            unique_contacts=("contact_hash", "nunique"),
            eligible_sentiment=("eligible_sentiment", "sum"),
            eligible_theme=("eligible_theme", "sum"),
            pii_detected=("pii_detected", "sum"),
            repeated_records=("repeated_text", "sum"),
            estimated_population=("sample_weight", "sum"),
        )
        .reset_index()
    )


def create_validation_sample(dataset: pd.DataFrame, random_state: int) -> pd.DataFrame:
    eligible = dataset.loc[dataset["eligible_sentiment"]].copy()
    if "sentiment" not in eligible.columns:
        return pd.DataFrame()
    eligible["validation_stratum"] = eligible["source_group"] + " | " + eligible["sentiment"].fillna("Sin resultado")
    sample = (
        eligible.groupby("validation_stratum", group_keys=False)
        .apply(lambda frame: frame.sample(min(len(frame), 35), random_state=random_state), include_groups=False)
        .reset_index(drop=True)
    )
    keep = [
        "record_id", "source_group", "source_detail", "created_at", "text_anonymized",
        "sentiment", "sentiment_confidence", "primary_theme", "primary_intent",
    ]
    sample = sample[keep]
    sample["human_sentiment"] = ""
    sample["human_theme"] = ""
    sample["review_notes"] = ""
    return sample


def main() -> None:
    args = parse_args()
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    social = read_source(args.social)
    surveys = read_source(args.surveys)
    conversations = read_source(args.conversations)
    weights = read_source(args.weights)

    dataset = build_unified_dataset(social, surveys, conversations, weights)
    dataset = add_quality_and_rules(dataset)

    sentiment_metadata = {"model": MODEL_NAME, "status": "not_run"}
    if args.skip_sentiment:
        dataset["sentiment"] = "Pendiente"
        dataset["sentiment_confidence"] = np.nan
        dataset["prob_negative"] = np.nan
        dataset["prob_neutral"] = np.nan
        dataset["prob_positive"] = np.nan
        dataset["sentiment_model"] = MODEL_NAME
        dataset["sentiment_model_commit"] = ""
        dataset["nps_text_contradiction"] = False
    else:
        dataset, sentiment_metadata = add_sentiment(
            dataset,
            batch_size=args.batch_size,
            confidence_threshold=args.confidence_threshold,
        )

    cluster_catalog = pd.DataFrame()
    if args.skip_clusters:
        dataset["discovery_cluster"] = pd.NA
        dataset["discovery_terms"] = ""
    else:
        dataset, cluster_catalog = add_discovery_clusters(
            dataset,
            n_clusters=args.clusters,
            random_state=args.random_state,
        )

    quality_summary = create_quality_summary(dataset)
    validation_sample = create_validation_sample(dataset, args.random_state)

    output_columns = [
        "record_id", "case_id", "contact_hash", "created_at", "month",
        "source_group", "source_detail", "source_channel", "channel",
        "survey_name", "nps_value", "sample_weight", "is_sample",
        "text_anonymized", "context_anonymized", "word_count", "text_type",
        "eligible_sentiment", "eligible_theme", "pii_detected", "context_pii_detected",
        "text_repetitions", "repeated_text", "possible_campaign",
        "primary_theme", "detected_themes", "discovery_cluster", "discovery_terms",
        "primary_intent", "detected_intents", "frustration_signal",
        "sentiment", "sentiment_confidence", "prob_negative", "prob_neutral",
        "prob_positive", "sentiment_model", "sentiment_model_commit",
        "nps_text_contradiction", "tags",
    ]
    dataset[output_columns].to_csv(
        output_dir / "dataset_escucha_ciudadana_procesado.csv",
        index=False,
        encoding="utf-8-sig",
    )
    quality_summary.to_csv(output_dir / "resumen_control_calidad.csv", index=False, encoding="utf-8-sig")
    cluster_catalog.to_csv(output_dir / "catalogo_clusters_descubrimiento.csv", index=False, encoding="utf-8-sig")
    validation_sample.to_csv(output_dir / "muestra_validacion_manual.csv", index=False, encoding="utf-8-sig")

    metadata = {
        "input_rows": {
            "social": len(social),
            "surveys": len(surveys),
            "conversations": len(conversations),
            "weights": len(weights),
        },
        "output_rows": len(dataset),
        "sentiment": sentiment_metadata,
        "parameters": vars(args),
    }
    (output_dir / "metadata_proceso.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    print(quality_summary.to_string(index=False))
    print(f"\nArchivos generados en: {output_dir.resolve()}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/postprocesar_para_metabase.py
from __future__ import annotations

import argparse
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


NO_THEME = "No identificada"

URL_RE = re.compile(r"(?:https?://|www\.)\S+", re.I)
WORD_RE = re.compile(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]+")

ATTACHMENT_RE = re.compile(
    r"^(?:ha enviado|envio|envió|adjunto|adjunta|compartio|compartió)\s+(?:una|un)?\s*"
    r"(?:ubicacion|ubicación|imagen|foto|documento|archivo|audio|video|sticker|gif|contacto)\b",
    re.I,
)

GREETING_RE = re.compile(
    r"^(?:(?:hola+|holi+s?|buen(?:os)?\s+d[ií]as?|buenas(?:\s+(?:tardes|noches))?)"
    r"(?:\s+(?:buen(?:os)?\s+d[ií]as?|buenas(?:\s+(?:tardes|noches))?))?"
    r"|(?:chau|adi[oó]s|hasta\s+luego))\W*$",
    re.I,
)

GREETING_NORMALIZED_RE = re.compile(
    r"^(?:(?:hola+|holi+s?)(?:\s+(?:buen(?:os)?\s+dias?|buenas(?:\s+(?:tardes|noches))?))?"
    r"|buen(?:os)?\s+dias?|buenas(?:\s+(?:tardes|noches))?|chau|adios|hasta\s+luego)$",
    re.I,
)

NO_COMMENT_TERMS = {
    "nada",
    "ninguno",
    "ninguna",
    "sin comentario",
    "sin comentarios",
    "no tengo comentarios",
    "no tengo comentario",
    "no hay comentarios",
    "no aplica",
    "n a",
    "text",
    "test",
    "prueba",
}

CONTROL_TERMS = {
    "si",
    "no",
    "ok",
    "okay",
    "oki",
    "okey",
    "dale",
    "listo",
    "continuar",
    "volver",
    "menu",
    "menú",
    "finalizar",
    "salir",
    "inicio",
    "reclamo",
    "reclamos",
    "consulta",
    "consultas",
    "info",
    "informacion",
    "información",
}

COURTESY_TERMS = {
    "gracias",
    "muchas gracias",
    "gracias igualmente",
    "no gracias",
    "por favor",
    "perdon",
    "perdón",
    "disculpa",
    "disculpe",
}


# Las categorías específicas se evalúan primero. Las categorías amplias de
# discurso público quedan al final para no tapar reclamos o servicios concretos.
THEME_PATTERNS = [
    (
        "Multas y liberación de vehículos",
        r"\bmulta(?:s)?\b|liberaci[oó]n.*veh[ií]cul|corral[oó]n|remisi[oó]n|acarreo|"
        r"tmf[_ -]|veh[ií]culo.*reten|auto.*reten|moto.*reten",
    ),
    ("Licencia de conducir", r"licencia.*conduc|carnet.*conduc|registro.*conduc"),
    (
        "Habilitaciones y comercio",
        r"habilit|hab[_ -]|grupo[_ -]?fisc|fiscalizaci[oó]n ambiental|instituto.*alimento",
    ),
    ("Tributos y TGI", r"\btgi\b|tribut|impuesto|tasa municipal|pago anticipado"),
    (
        "Alumbrado público",
        r"alumbr|luminaria|farola|poste.*luz|luz.*calle|luces.*calle|sector.*apag",
    ),
    (
        "Residuos y limpieza",
        r"residu|basura|contenedor|recolecci[oó]n|limpieza|barrido|recicla|microbasural",
    ),
    (
        "Calles y veredas",
        r"\bcalle(?:s)?\b|\bvereda(?:s)?\b|\bbache(?:s)?\b|\bpozo(?:s)?\b|cord[oó]n|"
        r"pavimento|calzada|adoqu[ií]n|avenida.*rot|asfalto",
    ),
    ("Arbolado", r"\b[aá]rbol(?:es)?\b|arbolado|\bpoda\b|\brama(?:s)?\b|\bra[ií]z|forestaci[oó]n"),
    (
        "Anegamiento y desagües",
        r"aneg|inund|desag[uü]e|alcantar|sumidero|pluvial|boca.*tormenta",
    ),
    (
        "Estacionamiento y tránsito",
        r"estaciona|tr[aá]nsito|sem[aá]foro|cochera|veh[ií]culo abandonado|corte.*calle|"
        r"circulaci[oó]n vehicular",
    ),
    ("Bicicletas", r"\bbici(?:s)?\b|bicicleta|mibici|ciclov[ií]a"),
    ("Ruidos molestos", r"\bruido(?:s)?\b|m[uú]sica fuerte|volumen alto"),
    (
        "Personas en situación de calle",
        r"situaci[oó]n.*calle|persona.*calle|sin techo|gente.*calle",
    ),
    (
        "Transporte público",
        r"colectiv|transporte p[uú]blico|\bsube\b|\btaxi(?:s)?\b|\brem[ií]s|\bparada(?:s)?\b|"
        r"l[ií]nea\s+\d+",
    ),
    (
        "Salud y bienestar animal",
        r"\bsalud\b|hospital|centro.*salud|vacuna|\banimal(?:es)?\b|veterin|mascota|"
        r"perro|gato|zoonosis",
    ),
    ("Seguridad", r"seguridad|\brobo(?:s)?\b|delinc|insegur|polic[ií]a|violencia"),
    (
        "Costanera, río y playas",
        r"costanera|r[ií]o paran[aá]|\bparan[aá]\b|\bplaya(?:s)?\b|la florida|\bisla(?:s)?\b|"
        r"parque acu[aá]tico|parqueacuatico|rambla catalunya|ramblacatalunya|balneario",
    ),
    (
        "Agenda, cultura y deportes",
        r"\bagenda\b|\bcultura\b|\bevento(?:s)?\b|festival|deporte|pileta|colectividad|"
        r"museo|teatro|recital|turismo|d[ií]a del ni[nñ]o",
    ),
    (
        "Obras y espacio público",
        r"\bobra(?:s)?\b|\bplaza(?:s)?\b|\bparque(?:s)?\b|rambla|espacio p[uú]blico|"
        r"juego infantil|mobiliario urbano",
    ),
    (
        "Barrios y territorio",
        r"\bbarrio(?:s)?\b|zona (?:sur|norte|oeste|centro)|\bdistrito(?:s)?\b|vecinal|territorio",
    ),
    (
        "MuniBot y atención digital",
        r"munibot|\bbot\b|whatsapp|p[aá]gina|sitio web|web municipal|perfil digital|firma digital|"
        r"atenci[oó]n (?:del bot|digital|virtual)|servicio del bot",
    ),
    (
        "Gestión municipal e Intendente",
        r"municipalidad|municipio|intendente|javkin|gesti[oó]n municipal|gobierno municipal|"
        r"muni rosario",
    ),
    (
        "Imagen e identidad de Rosario",
        r"amo rosario|rosario.{0,20}hermos|hermos.{0,20}rosario|mi ciudad|orgullo rosarin|"
        r"monumento.*bandera|cuna.*bandera|identidad rosarin|rosarin[oa]s?",
    ),
]


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Refina las salidas de escucha ciudadana sin volver a ejecutar el modelo."
    )
    parser.add_argument("--input", required=True, help="CSV procesado por el notebook")
    parser.add_argument("--output-dir", required=True)
    parser.add_argument("--random-state", type=int, default=42)
    return parser.parse_args()


def remove_accents(value: str) -> str:
    return "".join(
        char
        for char in unicodedata.normalize("NFD", str(value or ""))
        if unicodedata.category(char) != "Mn"
    )


def normalize_text(value: str) -> str:
    value = remove_accents(str(value or "").lower())
    value = URL_RE.sub(" [url] ", value)
    value = re.sub(r"[^a-z0-9ñ\[\]]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def classify_refined(source_group: str, text: str, normalized: str, word_count: int) -> str:
    raw = str(text or "").strip()
    if not normalized or not re.search(r"[a-z0-9ñ]", normalized):
        return "Vacío, emoji o puntuación"
    if ATTACHMENT_RE.search(raw) or re.fullmatch(
        r"(?:ubicacion|imagen|foto|documento|archivo|audio|video|sticker|gif|contacto)",
        normalized,
    ):
        return "Adjunto, ubicación o contacto"
    if normalized in {"email", "numero", "url", "[email]", "[numero]", "[url]"}:
        return "Dato o enlace sin opinión"
    if re.fullmatch(r"(?:\d+|[a-zñ])", normalized) or re.fullmatch(r"\d+(?: y \d+)+", normalized):
        return "Opción o control"
    if GREETING_RE.fullmatch(raw) or GREETING_NORMALIZED_RE.fullmatch(normalized):
        return "Saludo o despedida"
    if normalized in NO_COMMENT_TERMS:
        return "Sin comentario"
    if normalized in CONTROL_TERMS:
        return "Opción o control"
    if source_group == "Conversación privada" and normalized in {
        normalize_text(value) for value in COURTESY_TERMS
    }:
        return "Cortesía operativa"
    if normalized in {normalize_text(value) for value in COURTESY_TERMS}:
        return "Contenido breve"
    if "[url]" in normalized and word_count <= 3:
        return "Dato o enlace sin opinión"
    if word_count <= 2:
        return "Contenido breve"
    return "Contenido"


def detect_theme_list(normalized: str) -> list[str]:
    if not normalized:
        return []
    return [name for name, pattern in THEME_PATTERNS if re.search(pattern, normalized, flags=re.I)]


def choose_theme(
    source_group: str,
    text_type: str,
    word_count: int,
    text_themes: list[str],
    context_themes: list[str],
) -> tuple[str, str, bool, str]:
    usable_text = text_type in {"Contenido", "Contenido breve"}
    context_allowed = source_group != "Social público" and usable_text and (
        word_count >= 3 or (word_count >= 2 and text_type == "Contenido breve")
    )

    if text_themes:
        conflict = bool(context_themes and not set(text_themes).intersection(context_themes))
        origin = "Texto + contexto" if set(text_themes).intersection(context_themes) else "Texto"
        return text_themes[0], origin, conflict, " | ".join(text_themes)
    if context_allowed and context_themes:
        return context_themes[0], "Contexto/tags", False, " | ".join(context_themes)
    return NO_THEME, "No identificada", False, ""


def build_validation_sample(dataset: pd.DataFrame, random_state: int) -> pd.DataFrame:
    candidates = dataset.loc[dataset["eligible_sentiment_refined"]].copy()
    candidates["validation_stratum"] = (
        candidates["source_group"].astype(str)
        + " | "
        + candidates["sentiment_dashboard"].astype(str)
    )
    sample = (
        candidates.groupby("validation_stratum", group_keys=False)
        .apply(
            lambda frame: frame.sample(min(len(frame), 35), random_state=random_state),
            include_groups=False,
        )
        .reset_index(drop=True)
    )
    columns = [
        "record_id",
        "source_group",
        "source_detail",
        "created_at",
        "text_anonymized",
        "sentiment_dashboard",
        "sentiment_confidence",
        "primary_theme_dashboard",
        "theme_origin",
        "primary_intent",
    ]
    sample = sample[columns]
    sample["sentimiento_humano"] = ""
    sample["tematica_humana"] = ""
    sample["observaciones"] = ""
    return sample


def main() -> None:
    args = parse_args()
    input_path = Path(args.input)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    dataset = pd.read_csv(input_path, low_memory=False)
    dataset["text_anonymized"] = dataset["text_anonymized"].fillna("").astype(str)
    dataset["tags"] = dataset["tags"].fillna("").astype(str)
    dataset["sample_weight"] = pd.to_numeric(dataset["sample_weight"], errors="coerce").fillna(1.0)

    normalized_text = dataset["text_anonymized"].map(normalize_text)
    normalized_tags = dataset["tags"].map(normalize_text)
    word_count = dataset["text_anonymized"].map(lambda value: len(WORD_RE.findall(value)))

    dataset["word_count_refined"] = word_count.astype(int)
    dataset["text_type_refined"] = [
        classify_refined(source, text, normalized, int(words))
        for source, text, normalized, words in zip(
            dataset["source_group"],
            dataset["text_anonymized"],
            normalized_text,
            word_count,
        )
    ]
    # El postproceso no vuelve a ejecutar el modelo: nunca debe habilitar un
    # texto que la corrida original no puntuó.
    originally_scored = dataset["eligible_sentiment"].fillna(False).astype(bool)
    dataset["eligible_sentiment_refined"] = originally_scored & dataset[
        "text_type_refined"
    ].isin({"Contenido", "Contenido breve"})
    dataset["sentiment_dashboard"] = dataset["sentiment"]
    dataset.loc[~dataset["eligible_sentiment_refined"], "sentiment_dashboard"] = "No analizable"

    text_theme_lists = normalized_text.map(detect_theme_list)
    context_theme_lists = normalized_tags.map(detect_theme_list)
    chosen = [
        choose_theme(source, text_type, int(words), text_themes, context_themes)
        for source, text_type, words, text_themes, context_themes in zip(
            dataset["source_group"],
            dataset["text_type_refined"],
            word_count,
            text_theme_lists,
            context_theme_lists,
        )
    ]
    dataset["primary_theme_dashboard"] = [value[0] for value in chosen]
    dataset["theme_origin"] = [value[1] for value in chosen]
    dataset["theme_context_conflict"] = [value[2] for value in chosen]
    dataset["detected_themes_dashboard"] = [value[3] for value in chosen]
    dataset["eligible_theme_refined"] = dataset["primary_theme_dashboard"].ne(NO_THEME)

    survey_mask = dataset["source_group"].eq("Encuesta") & dataset["nps_value"].notna()
    nps = pd.to_numeric(dataset["nps_value"], errors="coerce")
    dataset["nps_text_contradiction_refined"] = False
    dataset.loc[
        survey_mask & nps.isin([4, 5]) & dataset["sentiment_dashboard"].eq("Negativo"),
        "nps_text_contradiction_refined",
    ] = True
    dataset.loc[
        survey_mask & nps.isin([1, 2]) & dataset["sentiment_dashboard"].eq("Positivo"),
        "nps_text_contradiction_refined",
    ] = True

    dataset["weighted_records"] = dataset["sample_weight"]
    dataset["dashboard_note"] = np.select(
        [
            dataset["source_group"].eq("Conversación privada"),
            dataset["source_group"].eq("Encuesta"),
            dataset["source_group"].eq("Social público"),
        ],
        [
            "Muestra ponderada de mensajes ciudadanos",
            "Comentarios completos de encuestas",
            "Comentarios públicos completos",
        ],
        default="",
    )

    metabase_columns = [
        "record_id",
        "case_id",
        "contact_hash",
        "created_at",
        "month",
        "source_group",
        "source_detail",
        "source_channel",
        "channel",
        "survey_name",
        "nps_value",
        "sample_weight",
        "weighted_records",
        "is_sample",
        "dashboard_note",
        "text_anonymized",
        "word_count_refined",
        "text_type_refined",
        "eligible_sentiment_refined",
        "sentiment_dashboard",
        "sentiment_confidence",
        "eligible_theme_refined",
        "primary_theme_dashboard",
        "detected_themes_dashboard",
        "theme_origin",
        "theme_context_conflict",
        "primary_intent",
        "frustration_signal",
        "text_repetitions",
        "repeated_text",
        "possible_campaign",
        "nps_text_contradiction_refined",
        "sentiment_model",
        "sentiment_model_commit",
    ]
    metabase = dataset[metabase_columns].copy()
    metabase_path = output_dir / "dataset_escucha_ciudadana_metabase.csv"
    metabase.to_csv(metabase_path, index=False, encoding="utf-8-sig")

    summary = (
        dataset.groupby("source_group", dropna=False)
        .agg(
            records=("record_id", "size"),
            estimated_records=("weighted_records", "sum"),
            eligible_sentiment=("eligible_sentiment_refined", "sum"),
            eligible_theme=("eligible_theme_refined", "sum"),
            unique_cases=("case_id", "nunique"),
            unique_contacts=("contact_hash", "nunique"),
        )
        .reset_index()
    )
    summary["sentiment_coverage_pct"] = (summary["eligible_sentiment"] / summary["records"] * 100).round(2)
    summary["theme_coverage_pct"] = (summary["eligible_theme"] / summary["records"] * 100).round(2)
    summary.to_csv(output_dir / "resumen_dataset_metabase.csv", index=False, encoding="utf-8-sig")

    validation = build_validation_sample(dataset, args.random_state)
    validation.to_csv(output_dir / "muestra_validacion_manual_refinada.csv", index=False, encoding="utf-8-sig")

    change_summary = {
        "input_rows": int(len(dataset)),
        "output_rows": int(len(metabase)),
        "newly_excluded_from_sentiment": int(
            (dataset["eligible_sentiment"].astype(bool) & ~dataset["eligible_sentiment_refined"]).sum()
        ),
        "theme_context_conflicts": int(dataset["theme_context_conflict"].sum()),
        "validation_rows": int(len(validation)),
        "files": {
            "metabase": metabase_path.name,
            "summary": "resumen_dataset_metabase.csv",
            "validation": "muestra_validacion_manual_refinada.csv",
        },
    }
    (output_dir / "metadata_postproceso.json").write_text(
        json.dumps(change_summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(summary.to_string(index=False))
    print(json.dumps(change_summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/pipeline_completo.py
"""Análisis censal de septiembre/octubre. Reutiliza reglas v2 sin tocarla."""
from __future__ import annotations

import argparse
from contextlib import closing
import json
import math
import sqlite3
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

if not (Path(__file__).parent / 'analisis_escucha_ciudadana_v2.py').exists():
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import analisis_escucha_ciudadana_v2 as core
import postprocesar_para_metabase as refined

START = '2026-09-01'
END = '2026-10-26'  # exclusivo
REVISION = 'a2cc0f67ebd705c55191e25a05ba23d885fcc09b'
PREDICTION_COLUMNS = ['text_normalized', 'sentiment_raw', 'sentiment_confidence',
                      'prob_negative', 'prob_neutral', 'prob_positive']


def select_period(frame, column, start=START, end=END, strict=False):
    dates = pd.to_datetime(frame[column], format='%d/%m/%Y %H:%M:%S', errors='coerce')
    if dates.isna().any():
        raise ValueError(f'Hay fechas inválidas en {column}')
    mask = dates.ge(pd.Timestamp(start)) & dates.lt(pd.Timestamp(end))
    if strict and not mask.all():
        raise ValueError('La exportación privada contiene mensajes fuera del período')
    return frame.loc[mask].reset_index(drop=True).copy()


def period_totals(totals):
    dates = pd.to_datetime(totals['mes'], format='%d/%m/%Y', errors='coerce')
    if dates.isna().any():
        raise ValueError('Hay fechas inválidas en los totales')
    return totals.loc[dates.ge(pd.Timestamp(START)) & dates.lt(pd.Timestamp(END))].reset_index(drop=True).copy()


def validate_coverage(private, totals):
    dates = pd.to_datetime(private['created_at'], format='%d/%m/%Y %H:%M:%S')
    actual = private.assign(month_key=dates.dt.strftime('%Y-%m')).groupby(['month_key', 'source_channel']).size()
    controls = period_totals(totals)
    controls['month_key'] = pd.to_datetime(controls['mes'], format='%d/%m/%Y').dt.strftime('%Y-%m')
    numbers = controls['mensajes_poblacion'].astype(str).str.replace('.', '', regex=False)
    controls['expected'] = pd.to_numeric(numbers, errors='raise')
    if controls[['month_key', 'source_channel']].duplicated().any():
        raise ValueError('Totales de cobertura duplicados')
    expected = controls.set_index(['month_key', 'source_channel'])['expected']
    comparison = pd.concat([actual.rename('actual'), expected], axis=1).fillna(-1)
    if comparison.empty or not comparison['actual'].eq(comparison['expected']).all():
        raise ValueError('No coincide la cobertura privada por mes/canal: ' + comparison.to_json(orient='split'))


def build_census_dataset(social, surveys, private, totals):
    social = select_period(social, 'created_at')
    surveys = select_period(surveys, 'sent_at')
    private = select_period(private, 'created_at', strict=True)
    for frame, key in [(social, 'activity_id'), (surveys, 'response_id'), (private, 'activity_id')]:
        if frame[key].isna().any() or frame[key].astype(str).str.strip().eq('').any():
            raise ValueError('Falta un identificador de registro')
        if frame[key].duplicated().any():
            raise ValueError('Hay identificadores duplicados en una fuente')
    validate_coverage(private, totals)
    frames = []
    definitions = [
        (social, 'SOC_', 'activity_id', 'created_at', 'Social público', 'texto', 'canal'),
        (surveys, 'ENC_', 'response_id', 'sent_at', 'Encuesta', 'comentario', 'source_channel'),
        (private, 'CON_', 'activity_id', 'created_at', 'Conversación privada', 'mensaje_ciudadano', 'channel'),
    ]
    for source, prefix, key, date, group, text, channel in definitions:
        out = pd.DataFrame({
            'record_id': prefix + source[key].astype(str), 'case_id': source['case_id'],
            'contact_hash': source['contacto_anonimo'],
            'created_at': pd.to_datetime(source[date], format='%d/%m/%Y %H:%M:%S'),
            'source_group': group, 'source_channel': source['source_channel'],
            'channel': source[channel], 'raw_text': source[text], 'tags': source['tags'],
            'sample_weight': 1.0, 'is_sample': False,
        })
        out['source_detail'] = (source['red_social'] + ' - ' + source['origen_publicacion']) if group == 'Social público' else (source['survey_name'] if group == 'Encuesta' else source['source_channel'])
        out['survey_name'] = source['survey_name'] if group == 'Encuesta' else ''
        out['nps_value'] = pd.to_numeric(source['nps_value'], errors='coerce') if group == 'Encuesta' else np.nan
        out['raw_context'] = source['contexto_bot_anterior'] if group == 'Conversación privada' else ''
        frames.append(out)
    result = pd.concat(frames, ignore_index=True)
    result['month'] = result['created_at'].dt.strftime('%Y-%m')
    return result


def cached_predictions(texts, predictor, path, namespace, batch_size):
    if batch_size < 1:
        raise ValueError('batch_size debe ser positivo')
    unique = list(dict.fromkeys(texts))
    if not unique:
        return pd.DataFrame(columns=PREDICTION_COLUMNS)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with closing(sqlite3.connect(path)) as db:
        db.execute('CREATE TABLE IF NOT EXISTS predictions (namespace TEXT, text TEXT, payload TEXT, PRIMARY KEY(namespace,text))')
        stored = {text: json.loads(payload) for text, payload in db.execute('SELECT text,payload FROM predictions WHERE namespace=?', (namespace,))}
        pending = [text for text in unique if text not in stored]
        print(f'Textos únicos: {len(unique):,}; recuperados: {len(unique)-len(pending):,}; pendientes: {len(pending):,}', flush=True)
        for offset in range(0, len(pending), batch_size):
            batch = pending[offset:offset+batch_size]
            rows = predictor(batch)
            if len(rows) != len(batch) or [row['text_normalized'] for row in rows] != batch:
                raise ValueError('El modelo devolvió un lote incompleto o desordenado')
            for row in rows:
                values = [float(row[name]) for name in PREDICTION_COLUMNS[2:]]
                if row['sentiment_raw'] not in {'POS', 'NEG', 'NEU'} or not all(math.isfinite(v) and 0 <= v <= 1 for v in values):
                    raise ValueError('Predicción inválida')
            with db:
                db.executemany('INSERT OR REPLACE INTO predictions VALUES (?,?,?)',
                               [(namespace, row['text_normalized'], json.dumps(row, ensure_ascii=False)) for row in rows])
            stored.update({row['text_normalized']: row for row in rows})
            if offset == 0 or (offset // batch_size) % 20 == 0 or offset+batch_size >= len(pending):
                print(f'Inferencia guardada: {min(offset+batch_size,len(pending)):,}/{len(pending):,}', flush=True)
        return pd.DataFrame([stored[text] for text in unique], columns=PREDICTION_COLUMNS)


def add_census_sentiment(dataset, cache_path, batch_size=64, threshold=0.55):
    # Mismo modelo y texto normalizado de la corrida previa. Revisión fijada.
    resources = {}
    def predict(texts):
        if not resources:
            import torch
            from transformers import AutoModelForSequenceClassification, AutoTokenizer
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            tokenizer = AutoTokenizer.from_pretrained(core.MODEL_NAME, revision=REVISION)
            model = AutoModelForSequenceClassification.from_pretrained(core.MODEL_NAME, revision=REVISION).to(device)
            model.eval()
            labels = [str(model.config.id2label[i]).upper() for i in range(model.config.num_labels)]
            if set(labels) != {'NEG', 'NEU', 'POS'}:
                raise ValueError('Etiquetas del modelo inesperadas: ' + str(labels))
            resources.update(torch=torch, device=device, tokenizer=tokenizer, model=model, labels=labels)
        torch = resources['torch']
        encoded = resources['tokenizer'](texts, padding=True, truncation=True, max_length=128, return_tensors='pt').to(resources['device'])
        with torch.inference_mode():
            probabilities = torch.softmax(resources['model'](**encoded).logits, dim=-1).cpu().numpy()
        result = []
        for text, values in zip(texts, probabilities):
            probs = dict(zip(resources['labels'], values.tolist()))
            label = max(probs, key=probs.get)
            result.append(dict(text_normalized=text, sentiment_raw=label, sentiment_confidence=probs[label],
                               prob_negative=probs['NEG'], prob_neutral=probs['NEU'], prob_positive=probs['POS']))
        return result
    eligible = dataset.loc[dataset['eligible_sentiment'], 'text_normalized'].tolist()
    table = cached_predictions(eligible, predict, cache_path, core.MODEL_NAME+'|'+REVISION+'|normalizado-v2-max128', batch_size)
    dataset = dataset.merge(table, on='text_normalized', how='left', validate='many_to_one')
    dataset['sentiment'] = dataset['sentiment_raw'].map({'POS': 'Positivo', 'NEG': 'Negativo', 'NEU': 'Neutro'})
    dataset.loc[~dataset['eligible_sentiment'], 'sentiment'] = 'No analizable'
    dataset.loc[dataset['eligible_sentiment'] & dataset['sentiment_confidence'].lt(threshold), 'sentiment'] = 'Incierto'
    if dataset.loc[dataset['eligible_sentiment'], 'sentiment'].isna().any():
        raise ValueError('Quedaron mensajes analizables sin predicción')
    dataset['sentiment_model'] = core.MODEL_NAME
    dataset['sentiment_model_commit'] = REVISION
    dataset['nps_text_contradiction'] = (
        dataset['source_group'].eq('Encuesta') & (
            (dataset['nps_value'].isin([4, 5]) & dataset['sentiment'].eq('Negativo')) |
            (dataset['nps_value'].isin([1, 2]) & dataset['sentiment'].eq('Positivo'))))
    return dataset, dict(model=core.MODEL_NAME, commit=REVISION,
                         device=str(resources.get('device', 'cache')), unique_texts_scored=len(table),
                         batch_size=batch_size, confidence_threshold=threshold)


def main():
    parser = argparse.ArgumentParser(description='Escucha ciudadana completa, septiembre y octubre 2026')
    parser.add_argument('--social', required=True)
    parser.add_argument('--surveys', required=True)
    parser.add_argument('--private', required=True)
    parser.add_argument('--totals', required=True, help='Control de cobertura, no ponderaciones')
    parser.add_argument('--output-dir', required=True)
    parser.add_argument('--prepare-only', action='store_true')
    parser.add_argument('--cache', default='cache_sentimiento_sep_oct_2026.sqlite')
    parser.add_argument('--batch-size', type=int, default=64)
    args = parser.parse_args()
    social, surveys, private, totals = [core.read_source(path) for path in [args.social, args.surveys, args.private, args.totals]]
    social = select_period(social, 'created_at')
    surveys = select_period(surveys, 'sent_at')
    private = select_period(private, 'created_at', strict=True)
    totals = period_totals(totals)
    dataset = build_census_dataset(social, surveys, private, totals)
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    manifest = dict(start_inclusive=START, end_exclusive=END, october_partial=True,
                    sampling=False, weighting=False, expected_total=len(dataset),
                    source_rows=dataset.groupby('source_group').size().to_dict(),
                    private_coverage=private.assign(month_key=dataset.loc[dataset['source_group'].eq('Conversación privada'), 'month'].to_numpy()).groupby(['month_key','source_channel']).size().to_dict())
    manifest['private_coverage'] = {month+'|'+channel: int(n) for (month,channel),n in manifest['private_coverage'].items()}
    if args.prepare_only:
        for frame, name in [(social,'social_publico.csv'),(surveys,'encuestas.csv'),(private,'privados_completos.csv'),(totals,'control_totales.csv')]:
            frame.to_csv(output/name,index=False,encoding='utf-8-sig')
        (output/'manifest_inputs.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
        print(json.dumps(manifest,ensure_ascii=False,indent=2))
        return
    print('Cobertura validada. Preparando reglas para todos los registros...', flush=True)
    dataset = core.add_quality_and_rules(dataset)
    # Excluir controles refinados ANTES del modelo; nunca sustituirlos por neutros.
    types = [refined.classify_refined(source,text,normalized,int(words)) for source,text,normalized,words in zip(
        dataset['source_group'],dataset['text_anonymized'],dataset['text_normalized'],dataset['word_count'])]
    dataset['eligible_sentiment'] &= pd.Series(types,index=dataset.index).isin({'Contenido','Contenido breve'})
    dataset, sentiment_metadata = add_census_sentiment(dataset,args.cache,args.batch_size)
    dataset['discovery_cluster'] = pd.NA
    dataset['discovery_terms'] = ''
    # No exportar texto/contexto crudos al dataset de resultados.
    processed = output/'dataset_escucha_ciudadana_procesado.csv'
    dataset.drop(columns=['raw_text','raw_context','text_normalized','tags_normalized','sentiment_raw']).to_csv(processed,index=False,encoding='utf-8-sig')
    core.create_quality_summary(dataset).to_csv(output/'resumen_control_calidad.csv',index=False,encoding='utf-8-sig')
    processor = Path(refined.__file__).resolve()
    subprocess.run([sys.executable,str(processor),'--input',str(processed),'--output-dir',str(output)],check=True)
    final_path = output/'dataset_escucha_ciudadana_metabase.csv'
    final = pd.read_csv(final_path,low_memory=False,dtype={'case_id':str,'contact_hash':str})
    if len(final)!=len(dataset) or final['record_id'].duplicated().any() or not final['weighted_records'].eq(1).all() or final['is_sample'].any():
        raise ValueError('El dataset final no cumple el conteo censal')
    final.loc[final['source_group'].eq('Conversación privada'),'dashboard_note'] = 'Mensajes privados completos del universo textual definido; sin ponderación'
    final.to_csv(final_path,index=False,encoding='utf-8-sig')
    manifest.update(status='sentiment_executed_pending_human_review',sentiment=sentiment_metadata,discovery_clusters='not_run',
                    note='Conteos de registros reales; no personas ni conversaciones únicas. Anonimización automática básica: revisar datos personales antes de compartir.')
    (output/'metadata_proceso_completo.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    print('Finalizado. Revisar la muestra humana antes de publicar.',flush=True)


if __name__ == '__main__':
    main()


## 4. Ejecutar el análisis completo

El control se repite antes de inferir: si falta algún mensaje privado, se detiene; no se compensa con pesos. El modelo procesa una vez cada texto normalizado repetido, pero se conservan **todos los registros** en los conteos.

Se guardan las predicciones por lotes en una caché. Si se interrumpe una celda y la sesión sigue viva, volver a ejecutarla reutiliza los lotes completados. Si aparece un error de memoria GPU, reducir BATCH_SIZE a 32 o 16 y volver a ejecutar esta celda.

Si Colab elimina el entorno, se pierden sus archivos: para conservar avances, detener la ejecución y descargar la caché con la celda opcional del final; subirla junto al ZIP en una nueva sesión. No hay conexión automática a Drive. El modelo puede tardar; no se garantiza que Colab gratuito complete todo en una sesión.

El descubrimiento exploratorio de clusters no se ejecuta en esta actualización; se mantienen las reglas existentes de temáticas. Todavía habrá que validar manualmente una muestra del resultado.


In [ ]:
import subprocess
import sys
from pathlib import Path
import os
os.chdir('/content')
BATCH_SIZE = 64
OUTPUT = Path('/content/salidas_escucha_sep_oct_2026')
command = [sys.executable, '-u', '/content/pipeline_completo.py',
           '--social', str(DATA/'social_publico.csv'),
           '--surveys', str(DATA/'encuestas.csv'),
           '--private', str(DATA/'privados_completos.csv'),
           '--totals', str(DATA/'control_totales.csv'),
           '--output-dir', str(OUTPUT), '--cache', str(CACHE),
           '--batch-size', str(BATCH_SIZE)]
subprocess.run(command, check=True)


## 5. Comprobar resultados

Todos los registros cuentan una vez. Los controles y saludos sin opinión pueden quedar como “No analizable”, y las predicciones de baja confianza como “Incierto”. Ninguno debe convertirse automáticamente en neutro. Los nombres históricos de las columnas de pesos se conservan solo por compatibilidad y deben valer 1.


In [ ]:
import pandas as pd
import json
dataset = pd.read_csv(OUTPUT/'dataset_escucha_ciudadana_metabase.csv', low_memory=False)
assert len(dataset) == 317325
assert not dataset['record_id'].duplicated().any()
assert dataset['sample_weight'].eq(1).all() and dataset['weighted_records'].eq(1).all()
assert not dataset['is_sample'].any()
assert dataset.groupby('source_group').size().to_dict() == manifest['source_rows']
dates = pd.to_datetime(dataset['created_at'])
assert dates.ge('2026-09-01').all() and dates.lt('2026-10-26').all()
assert dataset.loc[dataset['eligible_sentiment_refined'], 'sentiment_dashboard'].notna().all()
display(pd.read_csv(OUTPUT/'resumen_dataset_metabase.csv'))
display(pd.crosstab(dataset['source_group'], dataset['sentiment_dashboard']))
print('Controles aprobados. Pendiente: revisión humana y actualización del tablero.')


## 6. Descargar las salidas

Descargar el ZIP y compartirlo para revisar resultados, muestra humana y preparar la actualización de Metabase/app. **No subir todavía el dataset como resultado validado.** Los totales de sentimiento reflejan registros analizables, no necesariamente todo el volumen exportado.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/salidas_escucha_sep_oct_2026', 'zip', OUTPUT)
files.download(archive)


## Opcional: respaldar la caché

Ejecutar cuando la inferencia esté detenida o haya terminado. Permite conservar lotes completados si se cambia de entorno. La caché contiene texto normalizado con anonimización básica: tratarla como datos internos, no compartir públicamente.


In [ ]:
from google.colab import files
if CACHE.exists():
    files.download(str(CACHE))
else:
    print('Todavía no se creó una caché.')
